
# Binomial Model Notebook for Questions 1–7

This notebook solves the test questions using a **15-period binomial tree** calibrated to a **Black–Scholes geometric Brownian motion** with:

- $S_0 = 100$
- $K = 110$
- $T = 0.25$
- $r = 2\%$
- $\sigma = 30\%$
- dividend yield $c = 1\%$
- number of periods $N = 15$

The CRR-style binomial parameters are:

$$
\Delta t = \frac{T}{N}, \qquad
u = e^{\sigma\sqrt{\Delta t}}, \qquad
d = \frac{1}{u}
$$

$$
p = \frac{e^{(r-c)\Delta t} - d}{u-d}
$$

For American options, the value at each node is:

$$
V_{i,j} = \max(\text{exercise value}, \text{continuation value})
$$

where

$$
\text{continuation value}
=
e^{-r\Delta t}\left(pV_{i+1,j+1} + (1-p)V_{i+1,j}\right)
$$

The notebook also checks:
- whether early exercise is ever optimal,
- the earliest period where early exercise may happen,
- whether put-call parity holds for the American stock options,
- and the American call option on a **futures contract**.


In [ ]:
import math
import pandas as pd

# Base inputs
S0 = 100
K = 110
T = 0.25
r = 0.02
sigma = 0.30
c = 0.01
N = 15

dt = T / N
u = math.exp(sigma * math.sqrt(dt))
d = 1 / u
p = (math.exp((r - c) * dt) - d) / (u - d)
disc = math.exp(-r * dt)

summary = pd.DataFrame(
    {
        "parameter": ["dt", "u", "d", "p", "discount factor"],
        "value": [dt, u, d, p, disc],
    }
)

summary


## Helper functions

The first function prices an American call or put on the stock.

The second function prices an American option on a **futures contract** that expires later than the option.

For the futures price at node $(i,j)$, with futures expiry at period $N_f$:

$$
F_{i,j} = S_{i,j} e^{(r-c)(N_f-i)\Delta t}
$$

For an American call on futures, the immediate exercise value is:

$$
\max(F_{i,j}-K,0)
$$

because exercising gives the holder a long futures position and the marked-to-market gain is the difference between the current futures price and the strike.


In [ ]:
def american_option_on_stock(S0, K, r, c, sigma, T, N, option_type="call"):
    dt = T / N
    u = math.exp(sigma * math.sqrt(dt))
    d = 1 / u
    p = (math.exp((r - c) * dt) - d) / (u - d)
    disc = math.exp(-r * dt)

    # Terminal stock prices
    stock_prices = [S0 * (u ** j) * (d ** (N - j)) for j in range(N + 1)]

    # Terminal payoffs
    if option_type == "call":
        values = [max(s - K, 0.0) for s in stock_prices]
    else:
        values = [max(K - s, 0.0) for s in stock_prices]

    exercise_nodes = []

    # Backward induction
    for i in range(N - 1, -1, -1):
        new_values = []
        for j in range(i + 1):
            s_ij = S0 * (u ** j) * (d ** (i - j))
            continuation = disc * (p * values[j + 1] + (1 - p) * values[j])
            exercise = max(s_ij - K, 0.0) if option_type == "call" else max(K - s_ij, 0.0)
            value = max(exercise, continuation)

            if exercise > continuation + 1e-12 and exercise > 0:
                exercise_nodes.append(
                    {
                        "period": i,
                        "j": j,
                        "stock_price": s_ij,
                        "exercise_value": exercise,
                        "continuation_value": continuation,
                    }
                )

            new_values.append(value)

        values = new_values

    earliest = min((node["period"] for node in exercise_nodes), default=None)
    return {
        "price": values[0],
        "u": u,
        "d": d,
        "p": p,
        "exercise_nodes": exercise_nodes,
        "earliest_exercise_period": earliest,
    }


def american_call_on_futures(S0, K, r, c, sigma, T_total, N_futures, option_maturity_periods):
    dt = T_total / N_futures
    u = math.exp(sigma * math.sqrt(dt))
    d = 1 / u
    p = (math.exp((r - c) * dt) - d) / (u - d)
    disc = math.exp(-r * dt)

    def stock_price(i, j):
        return S0 * (u ** j) * (d ** (i - j))

    def futures_price(i, j):
        tau = (N_futures - i) * dt
        return stock_price(i, j) * math.exp((r - c) * tau)

    n = option_maturity_periods

    # Terminal option values at option maturity
    values = [max(futures_price(n, j) - K, 0.0) for j in range(n + 1)]

    exercise_nodes = []

    # Backward induction
    for i in range(n - 1, -1, -1):
        new_values = []
        for j in range(i + 1):
            F_ij = futures_price(i, j)
            continuation = disc * (p * values[j + 1] + (1 - p) * values[j])
            exercise = max(F_ij - K, 0.0)
            value = max(exercise, continuation)

            if exercise > continuation + 1e-12 and exercise > 0:
                exercise_nodes.append(
                    {
                        "period": i,
                        "j": j,
                        "futures_price": F_ij,
                        "exercise_value": exercise,
                        "continuation_value": continuation,
                    }
                )

            new_values.append(value)

        values = new_values

    earliest = min((node["period"] for node in exercise_nodes), default=None)
    return {
        "price": values[0],
        "u": u,
        "d": d,
        "p": p,
        "exercise_nodes": exercise_nodes,
        "earliest_exercise_period": earliest,
    }

## Question 1

Price the **American call option on the stock**.

In [ ]:
q1 = american_option_on_stock(S0, K, r, c, sigma, T, N, option_type="call")
q1["price"]


### Q1 result

The American call price is:

$$
\boxed{2.60}
$$

With these parameters, no early-exercise node appears in the tree, so the American call and European call are the same here.


## Question 2

Price the **American put option on the stock**.

In [ ]:
q2 = american_option_on_stock(S0, K, r, c, sigma, T, N, option_type="put")
q2["price"]


### Q2 result

The American put price is:

$$
\boxed{12.36}
$$

The put is deep enough in the money in some low-stock-price states that early exercise becomes optimal.


## Question 3

Is early exercise ever optimal for the American put from Question 2?

In [ ]:
len(q2["exercise_nodes"]) > 0


### Q3 result

$$
\boxed{\text{Yes}}
$$


## Question 4

What is the **earliest period** where early exercise might be optimal for the put?

In [ ]:
q2["earliest_exercise_period"]

In [ ]:
pd.DataFrame(q2["exercise_nodes"]).head(10)


### Q4 result

The earliest exercise period is:

$$
\boxed{5}
$$

So the first node where exercise can dominate continuation happens at **period 5**.


## Question 5

Do the stock option prices from Questions 1 and 2 satisfy put-call parity?

In [ ]:
C = q1["price"]
P = q2["price"]

lhs = C - P
rhs = S0 * math.exp(-c * T) - K * math.exp(-r * T)

pd.DataFrame(
    {
        "quantity": ["C - P", "S0*exp(-cT) - K*exp(-rT)", "difference"],
        "value": [lhs, rhs, lhs - rhs],
    }
)


### Q5 result

$$
\boxed{\text{No}}
$$

Standard put-call parity is an equality for **European** options.  
Here we are using **American** options, and the American put contains an early-exercise premium, so equality does not hold.


## Question 6

Price the **American call option on a futures contract** with:

- strike = 110
- option maturity = 10 periods
- futures expiry = 15 periods

In [ ]:
q6 = american_call_on_futures(
    S0=S0,
    K=K,
    r=r,
    c=c,
    sigma=sigma,
    T_total=T,
    N_futures=N,
    option_maturity_periods=10,
)
q6["price"]


### Q6 result

The American call on the futures contract is:

$$
\boxed{1.66}
$$


## Question 7

What is the **earliest period** in which the American futures call might be exercised early?

In [ ]:
q6["earliest_exercise_period"]

In [ ]:
pd.DataFrame(q6["exercise_nodes"]).head(10)


### Q7 result

The earliest exercise period is:

$$
\boxed{7}
$$



## Final answers summary

1. American call on stock: **2.60**  
2. American put on stock: **12.36**  
3. Is early exercise ever optimal for the put? **Yes**  
4. Earliest early-exercise period for the put: **5**  
5. Do the call and put satisfy put-call parity? **No**  
6. American call on futures: **1.66**  
7. Earliest early-exercise period for the futures call: **7**

---

## Notes

- The period numbering here is the natural tree numbering: period **0** is today, and period **15** is final maturity of the stock/futures tree.
- That is why an answer like **5** means “the first possible early-exercise node appears at period 5”.
- All final reported answers are rounded to **2 decimals** where applicable.
